In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 12.0 MB/s eta 0:00:00


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fWWH6uNiwsMyEIbGghuH")
project = rf.workspace("mohammad-zafar").project("warehouse-bzwra-miwku")
version = project.version(1)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to WAREHOUSE-1 in yolov8:: 100%|██████████| 3603/3603 [00:04<00:00, 740.74it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
print(dataset.location)
!cat {dataset.location}/data.yaml
!ls {dataset.location}

/content/WAREHOUSE-1
names:
- Acc
- Kain
- Kain-Acc-Person
- forklif
- hand-pallet
- person
nc: 6
roboflow:
  license: CC BY 4.0
  project: warehouse-bzwra-miwku
  url: https://app.roboflow.com/mohammad-zafar/warehouse-bzwra-miwku/1
  version: 1
  workspace: mohammad-zafar
test: ../test/images
train: ../train/images
val: ../valid/images
data.yaml  README.roboflow.txt	train  valid


In [4]:
import os
for s in ["train", "valid", "test"]:
    p = f"{dataset.location}/{s}/images"
    print(s, len(os.listdir(p)) if os.path.exists(p) else "missing")

train 1655
valid 145
test missing


In [5]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

In [6]:
results = model.train(data=f"{dataset.location}/data.yaml", epochs=30, imgsz=640, batch=16)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/WAREHOUSE-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimi

In [7]:
m = model.val()
print("mAP50:", m.box.map50, " mAP50-95:", m.box.map)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,266 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2986.6±971.1 MB/s, size: 88.6 KB)
val: Scanning /content/WAREHOUSE-1/valid/labels.cache... 145 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 145/145 43.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.6it/s 6.3s
                   all        145       2944       0.93      0.977      0.977      0.623
                   Acc        145       1560      0.986      0.974      0.991      0.569
                  Kain        145       1290      0.979      0.958      0.982       0.55
               forklif         24         24      0.997          1      0.995      0.738
           hand-pallet          5          5       0.77          1      0.962      0.728
                person         51        

In [9]:
model.track(source="video.mp4", tracker="bytetrack.yaml", save=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/203) /content/video.mp4: 384x640 1 Kain, 8.0ms
video 1/1 (frame 2/203) /content/video.mp4: 384x640 1 Kain, 16.6ms
video 1/1 (frame 3/203) /content/video.mp4: 384x640 1 Kain, 15.4ms
video 1/1 (frame 4/203) /content/video.mp4: 384x640 1 Kain, 15.0ms
video 1/1 (frame 5/203) /content/video.mp4: 384x640 1 Kain, 18.2ms
video 1/1 (frame 6/203) /content/video.mp4: 384x640 1 Kain, 14.8ms
video 1/1 (frame 7/203) /content/video.mp4: 384x640 1 Kain

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'Acc', 1: 'Kain', 2: 'Kain-Acc-Person', 3: 'forklif', 4: 'hand-pallet', 5: 'person'}
 obb: None
 orig_img: array([[[ 73,  65,  65],
         [ 95,  87,  87],
         [106,  98,  98],
         ...,
         [ 81,  70,  67],
         [ 77,  66,  63],
         [ 76,  65,  62]],
 
        [[ 89,  81,  81],
         [ 91,  83,  83],
         [ 97,  89,  89],
         ...,
         [ 77,  66,  63],
         [ 76,  65,  62],
         [ 82,  71,  68]],
 
        [[ 91,  83,  83],
         [ 79,  71,  71],
         [ 81,  73,  73],
         ...,
         [ 79,  68,  65],
         [ 77,  66,  63],
         [ 83,  72,  69]],
 
        ...,
 
        [[ 65,  25,   9],
         [ 65,  25,   9],
         [ 65,  25,   9],
         ...,
         [ 52,  49,  50],
         [ 53,  50,  51],
         [ 53,  50,  51]],
 
        [[ 65,  25,   

In [10]:
model.export(format="onnx")

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,598,266 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (5.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 520ms
Prepared 4 packages in 1.87s
Installed 4 packages in 195ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.31.0
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 2.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming w

'/content/runs/detect/train/weights/best.onnx'

In [11]:
!zip -r -q runs.zip runs

In [12]:
from google.colab import files
files.download("runs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>